<a href="https://colab.research.google.com/github/joelblakewasajja-design/LifeOS/blob/main/LifeOS_Nebius.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 LifeOS — Your Personal AI Operating System
Nebius x NVIDIA Global AI Hackathon · Track: **Personal AI**

Runs on **Nebius Token Factory** with an **NVIDIA Nemotron** open model.
One brain that understands plain language, manages your goals & tasks, plans your day, and listens to your voice.

**Setup:** Colab → 🔑 Secrets → add `NEBIUS_API_KEY` (from tokenfactory.nebius.com) → enable Notebook access.

In [1]:
!pip -q install openai gradio openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.2 MB/s eta 0:00:00


In [5]:
import os, json, re, datetime
from openai import OpenAI
from google.colab import userdata

NEBIUS_API_KEY = userdata.get("NEBIUS_API_KEY")

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=NEBIUS_API_KEY,
)

# Show the NVIDIA Nemotron models available to your key, then copy an ID into MODEL below
for m in client.models.list().data:
    if "nemotron" in m.id.lower():
        print(m.id)

SecretNotFoundError: Secret NEBIUS_API_KEY does not exist.

In [ ]:
# 👇 Paste an exact Nemotron ID from the list above
MODEL = "nvidia/Nemotron-3-Nano-30B-A3B"

r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say READY in one word."}],
    max_tokens=200,
)
print(r.choices[0].message.content)

In [ ]:
# ---------------- MEMORY ----------------
MEMORY_FILE = "/content/lifeos_memory.json"

def load_memory():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, encoding="utf-8") as f:
            return json.load(f)
    return {"goals": [], "tasks": []}

def save_memory():
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(memory, f, indent=2, ensure_ascii=False)

memory = load_memory()

def _exists(items, title):
    return any(i["title"].lower() == title.lower() for i in items)

def add_goal(title):
    title = title.strip()
    if not title: return "Goal cannot be empty."
    if _exists(memory["goals"], title): return f"🎯 Already have that goal: {title}"
    memory["goals"].append({"title": title, "completed": False}); save_memory()
    return f"🎯 Goal added: {title}"

def add_task(title):
    title = title.strip()
    if not title: return "Task cannot be empty."
    if _exists(memory["tasks"], title): return f"✅ Already have that task: {title}"
    memory["tasks"].append({"title": title, "completed": False}); save_memory()
    return f"✅ Task added: {title}"

def complete_task(number):
    try: i = int(number) - 1
    except: return "Enter a task number like 1 or 2."
    if not 0 <= i < len(memory["tasks"]): return "❌ That task number doesn't exist."
    memory["tasks"][i]["completed"] = True; save_memory()
    return f"🎉 Completed: {memory['tasks'][i]['title']}"

def complete_goal(number):
    try: i = int(number) - 1
    except: return "Enter a goal number like 1 or 2."
    if not 0 <= i < len(memory["goals"]): return "❌ That goal number doesn't exist."
    memory["goals"][i]["completed"] = True; save_memory()
    return f"🏆 Goal achieved: {memory['goals'][i]['title']}"

def delete_all_tasks():
    memory["tasks"] = []; save_memory()
    return "🗑️ All tasks deleted."

def memory_text():
    goals = "\n".join(f"{i}. [{'x' if g['completed'] else ' '}] {g['title']}" for i, g in enumerate(memory["goals"], 1)) or "None"
    tasks = "\n".join(f"{i}. [{'x' if t['completed'] else ' '}] {t['title']}" for i, t in enumerate(memory["tasks"], 1)) or "None"
    return f"GOALS:\n{goals}\n\nTASKS:\n{tasks}"

In [ ]:
# ---------------- AI BRAIN (Nebius + Nemotron) ----------------
def llm(messages, temperature=0.6, max_tokens=1200):
    r = client.chat.completions.create(model=MODEL, messages=messages,
                                       temperature=temperature, max_tokens=max_tokens)
    text = r.choices[0].message.content or ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()   # hide reasoning if the model emits it

SYSTEM = """You are LifeOS, a friendly personal AI operating system and life coach.
You manage the user's goals and tasks and help with school, work, projects, fitness, habits and everyday decisions.

Reply with ONLY a JSON object, no markdown fences:
{"actions": [...], "reply": "<your message to the user>"}

Allowed actions (use [] if none):
 {"type":"add_task","title":"..."}
 {"type":"add_goal","title":"..."}
 {"type":"complete_task","number":N}
 {"type":"complete_goal","number":N}
 {"type":"delete_all_tasks"}   <- ONLY if the user explicitly asks to delete/clear ALL tasks

Rules: only take an action when the user clearly asks or states a goal/task to track.
Numbers refer to the lists in LIFEOS MEMORY. "reply" is short, warm, practical, plain paragraphs,
mention what you did. Use their memory when relevant. If they're overwhelmed, keep it simple."""

history = []   # short conversation memory

def run_action(a):
    t = a.get("type")
    if t == "add_task":        return add_task(a.get("title", ""))
    if t == "add_goal":        return add_goal(a.get("title", ""))
    if t == "complete_task":   return complete_task(a.get("number"))
    if t == "complete_goal":   return complete_goal(a.get("number"))
    if t == "delete_all_tasks":return delete_all_tasks()
    return None

def ask_lifeos(message):
    message = (message or "").strip()
    if not message: return "Please type or say something first."
    now = datetime.datetime.now().strftime("%A %d %B %Y, %H:%M")
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "system", "content": f"Now: {now}\n\nLIFEOS MEMORY\n{memory_text()}"},
            *history[-6:],
            {"role": "user", "content": message}]
    try:
        raw = llm(msgs)
        m = re.search(r"\{.*\}", raw, flags=re.S)
        data = json.loads(m.group(0)) if m else {"actions": [], "reply": raw}
    except json.JSONDecodeError:
        data = {"actions": [], "reply": raw}
    except Exception as e:
        return f"⚠️ AI error: {type(e).__name__}: {e}"

    done = [r for r in (run_action(a) for a in data.get("actions", [])) if r]
    reply = data.get("reply", "").strip()
    history.extend([{"role": "user", "content": message}, {"role": "assistant", "content": reply}])
    return ("\n".join(done) + "\n\n" if done else "") + reply

def smart_plan(minutes=30):
    prompt = (f"Build a realistic plan for the time I have: {int(minutes)} minutes. "
              "Use my goals and unfinished tasks, order by priority, give each step a time slice, "
              "and end with one encouraging line. Reply as plain text (no JSON, no tables).")
    msgs = [{"role": "system", "content": "You are LifeOS, a practical personal planner."},
            {"role": "user", "content": f"LIFEOS MEMORY\n{memory_text()}\n\n{prompt}"}]
    try: return llm(msgs)
    except Exception as e: return f"⚠️ AI error: {type(e).__name__}: {e}"

def dashboard():
    g, t = memory["goals"], memory["tasks"]
    done = sum(x["completed"] for x in t)
    pct = round(done / len(t) * 100) if t else 0
    bar = "█" * (pct // 10) + "░" * (10 - pct // 10)
    return f"📊 LIFEOS DASHBOARD\n\n🎯 Goals: {sum(x['completed'] for x in g)}/{len(g)} achieved\n✅ Tasks: {done}/{len(t)} done\n📈 {bar} {pct}%\n\n{memory_text()}"

In [ ]:
print(ask_lifeos("I want to finish my hackathon project and I have a math test Friday. Add a task to review chapter 3."))

In [ ]:
# ---------------- VOICE (Whisper) ----------------
import whisper
whisper_model = whisper.load_model("base")

def voice_to_text(audio_path):
    if not audio_path: return ""
    return whisper_model.transcribe(audio_path)["text"].strip()

In [ ]:
# ---------------- APP ----------------
import gradio as gr

with gr.Blocks(title="LifeOS") as app:
    gr.Markdown("# 🧠 LifeOS\n### Your Personal AI Operating System · powered by NVIDIA Nemotron on Nebius")

    with gr.Tab("💬 AI Coach"):
        mic = gr.Audio(sources=["microphone"], type="filepath", label="🎙️ Speak (optional)")
        msg = gr.Textbox(label="Talk to LifeOS", lines=3,
                         placeholder="e.g. I have 30 minutes — what should I work on? / Add a task: study physics")
        out = gr.Textbox(label="LifeOS", lines=10)
        with gr.Row():
            send = gr.Button("🧠 Ask LifeOS", variant="primary")
            clear = gr.Button("Clear chat memory")
        mic.stop_recording(voice_to_text, mic, msg)
        send.click(ask_lifeos, msg, out)
        msg.submit(ask_lifeos, msg, out)
        clear.click(lambda: (history.clear() or "Chat memory cleared."), None, out)

    with gr.Tab("📅 Smart Plan"):
        mins = gr.Slider(10, 480, value=30, step=10, label="Time available (minutes)")
        plan_btn = gr.Button("📅 Plan my time", variant="primary")
        plan_out = gr.Textbox(label="Your plan", lines=14)
        plan_btn.click(smart_plan, mins, plan_out)

    with gr.Tab("🎯 Goals"):
        g_in = gr.Textbox(label="New goal"); g_btn = gr.Button("Add goal")
        g_num = gr.Textbox(label="Goal number to mark achieved"); g_done = gr.Button("🏆 Achieved")
        g_out = gr.Textbox(label="Result")
        g_btn.click(add_goal, g_in, g_out); g_done.click(complete_goal, g_num, g_out)

    with gr.Tab("✅ Tasks"):
        t_in = gr.Textbox(label="New task"); t_btn = gr.Button("Add task")
        t_num = gr.Textbox(label="Task number to complete"); t_done = gr.Button("🎉 Complete")
        t_out = gr.Textbox(label="Result")
        t_btn.click(add_task, t_in, t_out); t_done.click(complete_task, t_num, t_out)

    with gr.Tab("📊 Dashboard"):
        d_btn = gr.Button("🔄 Refresh"); d_out = gr.Textbox(label="Dashboard", lines=18)
        d_btn.click(dashboard, None, d_out)

app.launch(share=True, debug=True)